In [1]:
# Cell 1: Install required libraries and import everything
!pip install --upgrade numpy --quiet
!pip install transformers datasets optuna pyswarms deap torch --quiet

import numpy as np
print(f"NumPy version: {np.__version__}")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW  # Using torch.optim.AdamW
from datasets import load_dataset
from sklearn.metrics import f1_score, accuracy_score
import time
import gc
import random
from copy import deepcopy
import json
import warnings
warnings.filterwarnings('ignore')

# For GPU memory tracking
import subprocess

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"Memory Cached: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

# Fixed batch size
BATCH_SIZE = 32
MAX_LENGTH = 128  # Reduced for faster training during optimization

NumPy version: 2.4.6
Using device: cuda
GPU: Tesla T4
Memory Allocated: 0.00 GB
Memory Cached: 0.00 GB


In [2]:
# Cell 2: Load IMDB dataset with balanced splits, no overlap, no duplicates
print("Loading IMDB dataset...")

# Load full dataset with correct path
dataset = load_dataset("stanfordnlp/imdb")

# Function to remove duplicate texts while keeping first occurrence
def remove_duplicates(data):
    """Remove duplicate texts from dataset"""
    seen_texts = set()
    unique_indices = []

    for i, example in enumerate(data):
        text = example["text"]
        if text not in seen_texts:
            seen_texts.add(text)
            unique_indices.append(i)

    print(f"  Removed {len(data) - len(unique_indices)} duplicates from {len(data)} samples")
    return data.select(unique_indices)

# Function to balance dataset by label
def balance_dataset(data, n_samples):
    """Take n_samples total with equal positive and negative examples"""
    n_per_class = n_samples // 2

    pos_indices = [i for i, x in enumerate(data) if x["label"] == 1]
    neg_indices = [i for i, x in enumerate(data) if x["label"] == 0]

    # Shuffle indices
    random.shuffle(pos_indices)
    random.shuffle(neg_indices)

    # Take equal numbers from each class
    selected_pos = pos_indices[:n_per_class]
    selected_neg = neg_indices[:n_per_class]

    # Combine and shuffle
    all_indices = selected_pos + selected_neg
    random.shuffle(all_indices)

    return data.select(all_indices)

# Step 1: Remove duplicates from full train set
print("\nRemoving duplicates from training data...")
train_clean = remove_duplicates(dataset["train"])

# Step 2: Create balanced train set: 20,000 samples
print("\nCreating balanced training set (20,000 samples)...")
train_full = balance_dataset(train_clean, 20000)

# Step 3: Split into 90% train and 10% validation (balanced, no overlap)
print("\nSplitting into 90% train and 10% validation (balanced)...")
train_full = train_full.shuffle(seed=42)

# Split while maintaining balance
train_pos_indices = [i for i, x in enumerate(train_full) if x["label"] == 1]
train_neg_indices = [i for i, x in enumerate(train_full) if x["label"] == 0]

# 90% of each class for train
n_train_pos = int(len(train_pos_indices) * 0.9)
n_train_neg = int(len(train_neg_indices) * 0.9)

# These are index positions within train_full, NOT overlapping
train_indices = train_pos_indices[:n_train_pos] + train_neg_indices[:n_train_neg]
val_indices = train_pos_indices[n_train_pos:] + train_neg_indices[n_train_neg:]

# Shuffle indices
random.shuffle(train_indices)
random.shuffle(val_indices)

# Create final splits
train_dataset = train_full.select(train_indices)
val_dataset = train_full.select(val_indices)

# Step 4: Remove duplicates from test set and create balanced set
print("\nRemoving duplicates from test data...")
test_clean = remove_duplicates(dataset["test"])

# Also remove any texts that appear in training data
train_val_texts = set(train_dataset["text"]) | set(val_dataset["text"])
test_unique_indices = []
for i, example in enumerate(test_clean):
    if example["text"] not in train_val_texts:
        test_unique_indices.append(i)
print(f"  Removed {len(test_clean) - len(test_unique_indices)} test samples overlapping with train/val")
test_clean = test_clean.select(test_unique_indices)

print("\nCreating balanced test set (20,000 samples)...")
test_dataset = balance_dataset(test_clean, 20000)

# VERIFICATION
print(f"\n{'='*60}")
print(f"FINAL VERIFICATION:")
print(f"{'='*60}")

# Check sizes
print(f"\nTrain:      {len(train_dataset)} samples")
print(f"  Positive: {sum(1 for x in train_dataset if x['label'] == 1)}")
print(f"  Negative: {sum(1 for x in train_dataset if x['label'] == 0)}")
print(f"Validation: {len(val_dataset)} samples")
print(f"  Positive: {sum(1 for x in val_dataset if x['label'] == 1)}")
print(f"  Negative: {sum(1 for x in val_dataset if x['label'] == 0)}")
print(f"Test:       {len(test_dataset)} samples")
print(f"  Positive: {sum(1 for x in test_dataset if x['label'] == 1)}")
print(f"  Negative: {sum(1 for x in test_dataset if x['label'] == 0)}")

# Check overlap
train_texts = set(train_dataset["text"])
val_texts = set(val_dataset["text"])
test_texts = set(test_dataset["text"])

overlap_train_val = train_texts.intersection(val_texts)
overlap_train_test = train_texts.intersection(test_texts)
overlap_val_test = val_texts.intersection(test_texts)

print(f"\nOverlap Check:")
print(f"Train & Validation: {len(overlap_train_val)} samples",
      "✓ NO LEAKAGE!" if len(overlap_train_val) == 0 else "✗ WARNING!")
print(f"Train & Test:       {len(overlap_train_test)} samples",
      "✓ NO LEAKAGE!" if len(overlap_train_test) == 0 else "✗ WARNING!")
print(f"Val & Test:         {len(overlap_val_test)} samples",
      "✓ NO LEAKAGE!" if len(overlap_val_test) == 0 else "✗ WARNING!")

# Check balance
train_pos = sum(1 for x in train_dataset if x["label"] == 1)
train_neg = sum(1 for x in train_dataset if x["label"] == 0)
val_pos = sum(1 for x in val_dataset if x["label"] == 1)
val_neg = sum(1 for x in val_dataset if x["label"] == 0)
test_pos = sum(1 for x in test_dataset if x["label"] == 1)
test_neg = sum(1 for x in test_dataset if x["label"] == 0)

print(f"\nBalance Check:")
print(f"Train:      {train_pos}/{train_neg} (ratio: {train_pos/train_neg:.3f})")
print(f"Validation: {val_pos}/{val_neg} (ratio: {val_pos/val_neg:.3f})")
print(f"Test:       {test_pos}/{test_neg} (ratio: {test_pos/test_neg:.3f})")
print(f"{'='*60}")

Loading IMDB dataset...


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]


Removing duplicates from training data...
  Removed 96 duplicates from 25000 samples

Creating balanced training set (20,000 samples)...

Splitting into 90% train and 10% validation (balanced)...

Removing duplicates from test data...
  Removed 199 duplicates from 25000 samples
  Removed 91 test samples overlapping with train/val

Creating balanced test set (20,000 samples)...

FINAL VERIFICATION:

Train:      18000 samples
  Positive: 9000
  Negative: 9000
Validation: 2000 samples
  Positive: 1000
  Negative: 1000
Test:       20000 samples
  Positive: 10000
  Negative: 10000

Overlap Check:
Train & Validation: 0 samples ✓ NO LEAKAGE!
Train & Test:       0 samples ✓ NO LEAKAGE!
Val & Test:         0 samples ✓ NO LEAKAGE!

Balance Check:
Train:      9000/9000 (ratio: 1.000)
Validation: 1000/1000 (ratio: 1.000)
Test:       10000/10000 (ratio: 1.000)


In [3]:
# Cell 3: Tokenize all three datasets
print("Loading tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenization function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors=None  # Don't return tensors yet
    )

# Tokenize datasets
print("Tokenizing train dataset...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
print("Tokenizing validation dataset...")
val_dataset = val_dataset.map(tokenize_function, batched=True)
print("Tokenizing test dataset...")
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
columns = ["input_ids", "attention_mask", "label"]
train_dataset.set_format(type="torch", columns=columns)
val_dataset.set_format(type="torch", columns=columns)
test_dataset.set_format(type="torch", columns=columns)

print("\nTokenization complete!")

Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizing train dataset...


...

Tokenizing validation dataset...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing test dataset...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]


Tokenization complete!


In [ ]:
# Alternative fix
!pip install torchvision==0.18.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 136.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
import torch
import torch.onnx

# Mock the missing attribute if it doesn't exist
if not hasattr(torch.onnx, '_CAFFE2_ATEN_FALLBACK'):
    torch.onnx._CAFFE2_ATEN_FALLBACK = False

import torchvision.io
import sys

# Monkey patch torchvision.io to add VideoReader
if not hasattr(torchvision.io, 'VideoReader'):
    class VideoReaderStub:
        def __init__(self, *args, **kwargs):
            raise ImportError(
                "VideoReader is not available in this torchvision version. "
                "Consider using a workaround or downgrading torchvision."
            )
    torchvision.io.VideoReader = VideoReaderStub

print("✓ Applied torchvision VideoReader patch")

from datasets import Dataset

ds = Dataset.from_dict({"value": [1, 2, 3]})
ds.set_format("torch")
print("✓ Dataset formatting works:", ds[0])

In [ ]:
# Cell 5: Define training, evaluation, and metrics functions (with progress)
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

def get_gpu_memory_usage():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 ** 2)
    return 0

def reset_gpu_memory_stats():
    """Reset GPU memory stats"""
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

def evaluate_model(model, data_loader, desc="Evaluating"):
    """Evaluate model and return metrics"""
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0
    inference_times = []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc=desc, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            # Measure inference time per batch
            start_time = time.time()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            inference_times.append(time.time() - start_time)

            total_loss += outputs.loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    avg_loss = total_loss / len(data_loader)
    avg_inference_time = np.mean(inference_times)

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "f1": f1,
        "inference_time_per_batch": avg_inference_time
    }

def train_model(hyperparams, train_loader, val_loader, epochs, lr, weight_decay,
                warmup_ratio, dropout_rate, verbose=True):
    """Train DistilBERT with given hyperparameters and return metrics"""

    # Reset GPU memory tracking
    reset_gpu_memory_stats()
    torch.cuda.empty_cache()
    gc.collect()

    if verbose:
        print(f"Loading model with dropout={dropout_rate}...")

    # Initialize model with dropout rate
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2,
        dropout=dropout_rate,
        attention_dropout=dropout_rate
    ).to(device)

    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if verbose:
        print(f"Total parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")

    # Optimizer
    optimizer = AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # Scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    if verbose:
        print(f"\nTraining for {epochs} epochs ({total_steps} total steps, {warmup_steps} warmup steps)...")
        print(f"Learning rate: {lr}, Weight decay: {weight_decay}")
        print("-" * 50)

    # Training
    training_start = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False) if verbose else train_loader

        for batch_idx, batch in enumerate(progress_bar):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()

            # Update progress bar
            if verbose and isinstance(progress_bar, tqdm):
                progress_bar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})

        avg_epoch_loss = epoch_loss / len(train_loader)
        if verbose:
            print(f"Epoch {epoch+1}/{epochs} - Avg Loss: {avg_epoch_loss:.4f}")

    training_time = time.time() - training_start

    if verbose:
        print(f"\nTraining completed in {training_time:.2f} seconds")
        print("-" * 50)
        print("Evaluating on validation set...")

    # Evaluate on validation set
    val_metrics = evaluate_model(model, val_loader, desc="Validating")

    # Get GPU memory usage
    gpu_memory = get_gpu_memory_usage()

    if verbose:
        print(f"Validation Loss: {val_metrics['loss']:.4f}")
        print(f"Validation Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"Validation F1: {val_metrics['f1']:.4f}")
        print(f"GPU Memory: {gpu_memory:.2f} MB")

    # Clean up
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1": val_metrics["f1"],
        "training_time": training_time,
        "inference_time_per_batch": val_metrics["inference_time_per_batch"],
        "gpu_memory_mb": gpu_memory
    }

print("Functions defined with progress tracking!")

In [ ]:
# Cell 6: Train baseline model with default hyperparameters
print("Training baseline model with default hyperparameters...")
print(f"{'='*60}")

# Default hyperparameters
baseline_hyperparams = {
    "learning_rate": 2e-5,
    "epochs": 5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "dropout_rate": 0.1
}

print("Hyperparameters:")
for k, v in baseline_hyperparams.items():
    print(f"  {k}: {v}")

# Train baseline model
baseline_results = train_model(
    hyperparams=baseline_hyperparams,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=baseline_hyperparams["epochs"],
    lr=baseline_hyperparams["learning_rate"],
    weight_decay=baseline_hyperparams["weight_decay"],
    warmup_ratio=baseline_hyperparams["warmup_ratio"],
    dropout_rate=baseline_hyperparams["dropout_rate"],
    verbose=True
)

print(f"\n{'='*60}")
print("BASELINE RESULTS (Validation Set):")
print(f"{'='*60}")
print(f"Validation Loss:      {baseline_results['val_loss']:.4f}")
print(f"Validation Accuracy:  {baseline_results['val_accuracy']:.4f}")
print(f"Validation F1-Score:  {baseline_results['val_f1']:.4f}")
print(f"Training Time:        {baseline_results['training_time']:.2f} seconds")
print(f"Inference Time/Batch: {baseline_results['inference_time_per_batch']:.4f} seconds")
print(f"GPU Memory Used:      {baseline_results['gpu_memory_mb']:.2f} MB")
print(f"{'='*60}")

# Estimate optimization time
estimated_time_20_runs = baseline_results['training_time'] * 20
print(f"\n⚠ ESTIMATE: 20 runs would take ~{estimated_time_20_runs/60:.1f} minutes")
print(f"   With 3 algorithms (PS + GA + Grid Search): ~{estimated_time_20_runs*3/60:.1f} minutes")
print(f"\nConsider reducing epochs or dataset size if this is too long.")

In [ ]:
# Cell 7: Define unified hyperparameter search space for all algorithms
print("=" * 60)
print("HYPERPARAMETER SEARCH SPACE")
print("=" * 60)

# Search space definition
# Format: [lower_bound, upper_bound, scale_type]
search_space = {
    "learning_rate": {
        "low": 1e-6,
        "high": 5e-5,
        "scale": "log",
        "description": "Learning rate for AdamW optimizer"
    },
    "epochs": {
        "low": 2,
        "high": 4,
        "scale": "int",
        "description": "Number of training epochs"
    },
    "weight_decay": {
        "low": 1e-5,
        "high": 1e-1,
        "scale": "log",
        "description": "L2 regularization strength"
    },
    "warmup_ratio": {
        "low": 0.0,
        "high": 0.2,
        "scale": "linear",
        "description": "Fraction of steps for LR warmup"
    },
    "dropout_rate": {
        "low": 0.05,
        "high": 0.4,
        "scale": "linear",
        "description": "Dropout probability for attention and hidden layers"
    }
}

# Print search space
for param, bounds in search_space.items():
    print(f"\n{param}:")
    print(f"  Range: [{bounds['low']}, {bounds['high']}]")
    print(f"  Scale: {bounds['scale']}")
    print(f"  {bounds['description']}")

# Extract bounds for optimization algorithms
param_names = list(search_space.keys())
lb = [search_space[p]["low"] for p in param_names]   # Lower bounds
ub = [search_space[p]["high"] for p in param_names]   # Upper bounds
scales = [search_space[p]["scale"] for p in param_names]

print(f"\n{'='*60}")
print("Bounds arrays for algorithms:")
print(f"  Parameters: {param_names}")
print(f"  Lower bounds: {lb}")
print(f"  Upper bounds: {ub}")
print(f"  Scales: {scales}")
print(f"{'='*60}")

# Number of evaluations per algorithm
N_EVALUATIONS = 20
print(f"\n⚠ Each algorithm will run {N_EVALUATIONS} evaluations")
print(f"   Total across 3 algorithms: {N_EVALUATIONS * 3} training runs")

In [ ]:
# Cell 8: Lightweight training function for hyperparameter optimization
def train_model_fast(epochs, lr, weight_decay, warmup_ratio, dropout_rate):
    """Train DistilBERT silently and return metrics dictionary"""

    # Reset GPU memory tracking
    reset_gpu_memory_stats()
    torch.cuda.empty_cache()
    gc.collect()

    # Initialize model
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=2,
        dropout=dropout_rate,
        attention_dropout=dropout_rate
    ).to(device)

    # Optimizer
    optimizer = AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # Scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # Training
    training_start = time.time()

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()

    training_time = time.time() - training_start

    # Evaluate on validation set
    val_metrics = evaluate_model(model, val_loader, desc="")

    # GPU memory
    gpu_memory = get_gpu_memory_usage()

    # Clean up
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_f1": val_metrics["f1"],
        "training_time": training_time,
        "inference_time_per_batch": val_metrics["inference_time_per_batch"],
        "gpu_memory_mb": gpu_memory
    }

# Quick test
print("Testing fast training function...")
test_result = train_model_fast(epochs=1, lr=2e-5, weight_decay=0.01, warmup_ratio=0.1, dropout_rate=0.1)
print(f"Test run - Loss: {test_result['val_loss']:.4f}, Acc: {test_result['val_accuracy']:.4f}, Time: {test_result['training_time']:.2f}s")
print("✓ Fast training function ready!")

In [ ]:
# Cell 9: Particle Swarm Optimization for Hyperparameter Tuning
import pyswarms as ps

print("=" * 60)
print("PARTICLE SWARM OPTIMIZATION (PSO)")
print("=" * 60)

# PSO Settings
n_particles = 10
n_iterations = 2  # 10 particles × 2 iterations = 20 evaluations
options = {'c1': 0.5, 'c2': 0.3, 'w': 0.9}

print(f"Settings: {n_particles} particles × {n_iterations} iterations = {n_particles * n_iterations} evaluations")
print(f"Options: c1={options['c1']}, c2={options['c2']}, w={options['w']}")

# Storage for all evaluated configurations
pso_all_results = []

# Objective function for PSO
def pso_objective(particles):
    """PSO objective: minimize validation loss (maximize negative f1)"""
    n_particles = particles.shape[0]
    scores = np.zeros(n_particles)

    for i in range(n_particles):
        lr = particles[i, 0]
        epochs = int(particles[i, 1])
        wd = particles[i, 2]
        warmup = particles[i, 3]
        dropout = particles[i, 4]

        print(f"  PSO eval {i+1}/{n_particles}: lr={lr:.2e}, epochs={epochs}, wd={wd:.2e}, warmup={warmup:.3f}, dropout={dropout:.3f}")

        try:
            result = train_model_fast(
                epochs=epochs, lr=lr, weight_decay=wd,
                warmup_ratio=warmup, dropout_rate=dropout
            )
            # Objective: minimize validation loss (you can use -f1 for maximizing f1)
            scores[i] = result["val_loss"]

            # Store all results
            pso_all_results.append({
                "lr": lr, "epochs": epochs, "weight_decay": wd,
                "warmup_ratio": warmup, "dropout_rate": dropout,
                **result
            })

        except Exception as e:
            print(f"  ✗ Error: {e}")
            scores[i] = 999.0

    return scores

# Define bounds
bounds = (np.array(lb), np.array(ub))

# Run PSO
print("\nRunning PSO optimization...")
pso_start = time.time()

optimizer = ps.single.GlobalBestPSO(
    n_particles=n_particles,
    dimensions=5,
    options=options,
    bounds=bounds
)

best_cost, best_pos = optimizer.optimize(pso_objective, iters=n_iterations)
pso_time = time.time() - pso_start

# Best results
pso_best = {
    "algorithm": "PSO",
    "learning_rate": best_pos[0],
    "epochs": int(best_pos[1]),
    "weight_decay": best_pos[2],
    "warmup_ratio": best_pos[3],
    "dropout_rate": best_pos[4],
    "best_val_loss": best_cost,
    "optimization_time": pso_time
}

print(f"\n{'='*60}")
print("PSO BEST RESULTS:")
print(f"{'='*60}")
for k, v in pso_best.items():
    if isinstance(v, float) and k != "algorithm":
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")
print(f"{'='*60}")

# Find best by F1-score
best_by_f1 = max(pso_all_results, key=lambda x: x['val_f1'])
print(f"\nBest by F1-Score: {best_by_f1['val_f1']:.4f} (Loss: {best_by_f1['val_loss']:.4f})")

In [ ]:
# Cell 10: Genetic Algorithm for Hyperparameter Tuning
from deap import base, creator, tools, algorithms

print("=" * 60)
print("GENETIC ALGORITHM (GA)")
print("=" * 60)

# Clear previous DEAP setup if exists
if 'FitnessMin' in creator.__dict__:
    del creator.FitnessMin
if 'Individual' in creator.__dict__:
    del creator.Individual

# GA Settings
POP_SIZE = 10
N_GENERATIONS = 2  # 10 population × 2 generations = 20 evaluations
CX_PROB = 0.7
MUT_PROB = 0.3

print(f"Settings: Population={POP_SIZE} × Generations={N_GENERATIONS} = {POP_SIZE * N_GENERATIONS} evaluations")
print(f"Crossover prob={CX_PROB}, Mutation prob={MUT_PROB}")

# Storage
ga_all_results = []

# Create fitness and individual
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))  # Minimize loss
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Hyperparameter generators
toolbox.register("attr_lr", lambda: 10 ** np.random.uniform(np.log10(lb[0]), np.log10(ub[0])))
toolbox.register("attr_epochs", lambda: np.random.randint(lb[1], ub[1] + 1))
toolbox.register("attr_wd", lambda: 10 ** np.random.uniform(np.log10(lb[2]), np.log10(ub[2])))
toolbox.register("attr_warmup", lambda: np.random.uniform(lb[3], ub[3]))
toolbox.register("attr_dropout", lambda: np.random.uniform(lb[4], ub[4]))

# Individual and population
toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.attr_lr, toolbox.attr_epochs, toolbox.attr_wd,
                  toolbox.attr_warmup, toolbox.attr_dropout), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Evaluation function
def ga_evaluate(individual):
    lr, epochs, wd, warmup, dropout = individual
    epochs = int(epochs)

    print(f"  GA eval: lr={lr:.2e}, epochs={epochs}, wd={wd:.2e}, warmup={warmup:.3f}, dropout={dropout:.3f}")

    try:
        result = train_model_fast(
            epochs=epochs, lr=lr, weight_decay=wd,
            warmup_ratio=warmup, dropout_rate=dropout
        )

        ga_all_results.append({
            "lr": lr, "epochs": epochs, "weight_decay": wd,
            "warmup_ratio": warmup, "dropout_rate": dropout,
            **result
        })

        return (result["val_loss"],)
    except Exception as e:
        print(f"  ✗ Error: {e}")
        return (999.0,)

# Crossover (blend crossover)
def ga_crossover(ind1, ind2):
    for i in range(len(ind1)):
        alpha = np.random.random()
        ind1[i], ind2[i] = ind1[i] + alpha * (ind2[i] - ind1[i]), ind2[i] + alpha * (ind1[i] - ind2[i])

    # Fix epochs to int
    ind1[1] = int(round(np.clip(ind1[1], lb[1], ub[1])))
    ind2[1] = int(round(np.clip(ind2[1], lb[1], ub[1])))

    # Clip all values
    for ind in [ind1, ind2]:
        ind[0] = np.clip(ind[0], lb[0], ub[0])
        ind[2] = np.clip(ind[2], lb[2], ub[2])
        ind[3] = np.clip(ind[3], lb[3], ub[3])
        ind[4] = np.clip(ind[4], lb[4], ub[4])

    return ind1, ind2

# Mutation
def ga_mutate(individual):
    idx = np.random.randint(0, 5)
    if idx == 0:
        individual[idx] = 10 ** np.random.uniform(np.log10(lb[0]), np.log10(ub[0]))
    elif idx == 1:
        individual[idx] = np.random.randint(lb[1], ub[1] + 1)
    elif idx == 2:
        individual[idx] = 10 ** np.random.uniform(np.log10(lb[2]), np.log10(ub[2]))
    elif idx == 3:
        individual[idx] = np.random.uniform(lb[3], ub[3])
    elif idx == 4:
        individual[idx] = np.random.uniform(lb[4], ub[4])
    return (individual,)

# Register operators
toolbox.register("evaluate", ga_evaluate)
toolbox.register("mate", ga_crossover)
toolbox.register("mutate", ga_mutate)
toolbox.register("select", tools.selTournament, tournsize=3)

# Run GA
print("\nRunning Genetic Algorithm...")
population = toolbox.population(n=POP_SIZE)
ga_start = time.time()

for gen in range(N_GENERATIONS):
    print(f"\n--- Generation {gen+1}/{N_GENERATIONS} ---")

    # Evaluate
    fitnesses = list(map(toolbox.evaluate, population))
    for ind, fit in zip(population, fitnesses):
        ind.fitness.values = fit

    best_ind = tools.selBest(population, 1)[0]
    print(f"  Best loss: {best_ind.fitness.values[0]:.4f}")

    # Select and breed
    offspring = toolbox.select(population, len(population))
    offspring = list(map(toolbox.clone, offspring))

    # Crossover
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if np.random.random() < CX_PROB:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    # Mutation
    for mutant in offspring:
        if np.random.random() < MUT_PROB:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    population[:] = offspring

ga_time = time.time() - ga_start

# Best results
best_ga_ind = tools.selBest(population, 1)[0]
ga_best = {
    "algorithm": "GA",
    "learning_rate": best_ga_ind[0],
    "epochs": int(best_ga_ind[1]),
    "weight_decay": best_ga_ind[2],
    "warmup_ratio": best_ga_ind[3],
    "dropout_rate": best_ga_ind[4],
    "best_val_loss": best_ga_ind.fitness.values[0],
    "optimization_time": ga_time
}

print(f"\n{'='*60}")
print("GA BEST RESULTS:")
print(f"{'='*60}")
for k, v in ga_best.items():
    if isinstance(v, float) and k != "algorithm":
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")
print(f"{'='*60}")

# Best by F1
best_ga_f1 = max(ga_all_results, key=lambda x: x['val_f1'])
print(f"\nBest by F1-Score: {best_ga_f1['val_f1']:.4f} (Loss: {best_ga_f1['val_loss']:.4f})")

In [ ]:
# Cell 11: Grid Search for Hyperparameter Tuning
from itertools import product

print("=" * 60)
print("GRID SEARCH")
print("=" * 60)

# Define grid points for each hyperparameter
grid_lr = [1e-6, 5e-6, 1e-5, 3e-5, 5e-5]           # 5 points
grid_epochs = [2, 3, 4]                               # 3 points
grid_wd = [1e-5, 1e-3, 1e-1]                         # 3 points
grid_warmup = [0.0, 0.1, 0.2]                        # 3 points
grid_dropout = [0.1, 0.2, 0.3]                       # 3 points

# Total combinations would be 5×3×3×3×3 = 405 (too many!)
# Let's use random sampling from grid: 20 random combinations
np.random.seed(42)

grid_combinations = []
for _ in range(N_EVALUATIONS):
    combo = {
        "lr": np.random.choice(grid_lr),
        "epochs": int(np.random.choice(grid_epochs)),
        "wd": np.random.choice(grid_wd),
        "warmup": np.random.choice(grid_warmup),
        "dropout": np.random.choice(grid_dropout)
    }
    grid_combinations.append(combo)

print(f"Grid points per parameter: lr={len(grid_lr)}, epochs={len(grid_epochs)}, wd={len(grid_wd)}, warmup={len(grid_warmup)}, dropout={len(grid_dropout)}")
print(f"Total possible combinations: {len(grid_lr) * len(grid_epochs) * len(grid_wd) * len(grid_warmup) * len(grid_dropout)}")
print(f"Randomly sampled: {len(grid_combinations)} evaluations")

# Storage
grid_all_results = []

# Run Grid Search
print("\nRunning Grid Search...")
grid_start = time.time()

for i, combo in enumerate(grid_combinations):
    print(f"  Grid eval {i+1}/{N_EVALUATIONS}: lr={combo['lr']:.2e}, epochs={combo['epochs']}, wd={combo['wd']:.2e}, warmup={combo['warmup']:.3f}, dropout={combo['dropout']:.3f}")

    try:
        result = train_model_fast(
            epochs=combo["epochs"], lr=combo["lr"],
            weight_decay=combo["wd"], warmup_ratio=combo["warmup"],
            dropout_rate=combo["dropout"]
        )

        grid_all_results.append({
            "lr": combo["lr"], "epochs": combo["epochs"],
            "weight_decay": combo["wd"], "warmup_ratio": combo["warmup"],
            "dropout_rate": combo["dropout"], **result
        })

    except Exception as e:
        print(f"  ✗ Error: {e}")

grid_time = time.time() - grid_start

# Best results (by loss)
best_grid = min(grid_all_results, key=lambda x: x['val_loss'])
grid_best = {
    "algorithm": "Grid Search",
    "learning_rate": best_grid["lr"],
    "epochs": best_grid["epochs"],
    "weight_decay": best_grid["weight_decay"],
    "warmup_ratio": best_grid["warmup_ratio"],
    "dropout_rate": best_grid["dropout_rate"],
    "best_val_loss": best_grid["val_loss"],
    "optimization_time": grid_time
}

print(f"\n{'='*60}")
print("GRID SEARCH BEST RESULTS:")
print(f"{'='*60}")
for k, v in grid_best.items():
    if isinstance(v, float) and k != "algorithm":
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")
print(f"{'='*60}")

# Best by F1
best_grid_f1 = max(grid_all_results, key=lambda x: x['val_f1'])
print(f"\nBest by F1-Score: {best_grid_f1['val_f1']:.4f} (Loss: {best_grid_f1['val_loss']:.4f})")

In [ ]:
# Cell 12: Compare all optimization algorithms
print("=" * 70)
print("COMPARISON OF ALL OPTIMIZATION ALGORITHMS")
print("=" * 70)

# Collect best results from each algorithm
all_best = [pso_best, ga_best, grid_best]

print(f"\n{'Algorithm':<15} {'LR':<12} {'Epochs':<8} {'Weight Decay':<14} {'Warmup':<10} {'Dropout':<10} {'Val Loss':<12} {'Val F1':<10} {'Time (s)':<10}")
print("-" * 95)

for result in all_best:
    print(f"{result['algorithm']:<15} {result['learning_rate']:<12.2e} {result['epochs']:<8} "
          f"{result['weight_decay']:<14.2e} {result['warmup_ratio']:<10.3f} {result['dropout_rate']:<10.3f} "
          f"{result['best_val_loss']:<12.4f} {'N/A':<10} {result['optimization_time']:<10.1f}")

# Add baseline comparison
print(f"\n{'Baseline':<15} {2e-5:<12.2e} {5:<8} {0.01:<14.2e} {0.1:<10.3f} {0.1:<10.3f} "
      f"{baseline_results['val_loss']:<12.4f} {baseline_results['val_f1']:<10.4f} {baseline_results['training_time']:<10.1f}")

print("-" * 95)

# Find best overall
best_overall_loss = min(all_best, key=lambda x: x['best_val_loss'])
print(f"\n🏆 Best by Validation Loss: {best_overall_loss['algorithm']}")
print(f"   Loss: {best_overall_loss['best_val_loss']:.4f}")

print(f"\n{'='*70}")